# Action Recognition Thesis — Google Colab (Pro) Notebook

**Why Colab Pro:** up to ~24 h runtimes (vs Kaggle's 12 h), background execution, and bigger GPUs — so all three models can train in one session.

**Before running:**
1. **Runtime → Change runtime type → L4 GPU** (A100 also fine; high-RAM is automatic on the L4).
2. Add your **Kaggle API token as a Colab Secret** (secure — never goes in a cell):
   - Get a token: kaggle.com → **Settings → API → Create New Token** (copy the `KGAT_...` value).
   - In Colab, click the **🔑 key icon** in the left sidebar → **Add new secret** → Name: `KAGGLE_API_TOKEN`, Value: *(paste the token)* → enable **Notebook access**.

Run Cells 1–7 for **UCF101**, then the **HMDB51** section at the bottom.
All logic lives in `run_all.py` in the repo, so this notebook never needs editing.
Results persist to your Google Drive, so a dropped session resumes where it left off.

In [ ]:
# ── Cell 1: GPU check + mount Google Drive ──────────────────────────────────
import torch
print(f'PyTorch {torch.__version__}  CUDA={torch.cuda.is_available()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'  {p.name}  ({p.total_memory/1e9:.1f} GB)')

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 2: Install dependencies ────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'decord', 'fvcore', 'scikit-learn', 'huggingface_hub', 'kaggle'
], check=True)
print('deps installed')

In [ ]:
# ── Cell 3: Kaggle API auth via Colab Secrets (secure) ──────────────────────
# Requires a secret named KAGGLE_API_TOKEN (see the notebook header). The token
# is read from Colab's secret store -- it never appears in this cell or output.
import os
from google.colab import userdata

token = userdata.get('KAGGLE_API_TOKEN')
os.environ['KAGGLE_API_TOKEN'] = token
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(token.strip())
os.chmod('/root/.kaggle/access_token', 0o600)
print('kaggle auth configured from Colab secret')

In [ ]:
# ── Cell 4: Get UCF101 (Drive-cached zip -> unzip to fast local disk) ────────
import os, glob, shutil, subprocess

DATA_DIR  = '/content/ucf101'                              # fast local SSD (train reads here)
DRIVE_ZIP = '/content/drive/MyDrive/ucf101_dataset.zip'    # cached download
SLUG = 'matthewjansen/ucf101-action-recognition'

if not (os.path.isdir(DATA_DIR) and os.listdir(DATA_DIR)):
    if not os.path.exists(DRIVE_ZIP):
        print('Downloading UCF101 from Kaggle (~6.5 GB, one time)...')
        subprocess.run(['kaggle', 'datasets', 'download', '-d', SLUG, '-p', '/content', '--quiet'], check=True)
        local_zip = glob.glob('/content/ucf101-action-recognition*.zip')[0]
        print('Caching zip to Drive for future sessions...')
        shutil.copy(local_zip, DRIVE_ZIP)
        zip_to_use = local_zip
    else:
        print('Using cached dataset zip from Drive.')
        zip_to_use = DRIVE_ZIP
    os.makedirs(DATA_DIR, exist_ok=True)
    print('Unzipping to local disk...')
    subprocess.run(['unzip', '-q', '-o', zip_to_use, '-d', DATA_DIR], check=True)
print('data ready at', DATA_DIR)
subprocess.run(['ls', DATA_DIR])

In [ ]:
# ── Cell 5: Clone (or sync) the repo; results persist to Drive ──────────────
import subprocess, sys, os

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
# Persist results to Drive so the run resumes across sessions (skip-guard).
os.environ['RESULTS_DIR'] = '/content/drive/MyDrive/thesis_results'

REPO = 'https://github.com/AndreiBrehuescu/action-recognition-thesis.git'
WORK = '/content/thesis'
if not os.path.exists(WORK):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, WORK], check=True)
else:
    subprocess.run(['git', '-C', WORK, 'fetch', '--depth', '1', 'origin', 'master'], check=True)
    subprocess.run(['git', '-C', WORK, 'reset', '--hard', 'origin/master'], check=True)
os.chdir(WORK)
subprocess.run(['git', '-C', WORK, 'log', '-1', '--oneline'])
print('repo ready at', WORK, '| results ->', os.environ['RESULTS_DIR'])

In [ ]:
# ── Cell 6: UCF101 — train -> benchmark -> Pareto plot ──────────────────────
# All settings live in run_all.py; edit there (in the repo), not here.
# Resumable: finished models are skipped on re-run (results are on Drive).
import subprocess, sys
subprocess.run([sys.executable, 'run_all.py',
    '--data-root', '/content/ucf101',
    '--dataset', 'ucf101',
    '--epochs', '10',
    '--batch-size', '8',
], check=True)

In [ ]:
# ── Cell 7: Show the UCF101 Pareto plot ─────────────────────────────────────
from IPython.display import Image, display
display(Image('/content/drive/MyDrive/thesis_results/pareto_ucf101.png'))

---
## HMDB51 (secondary dataset)

Run Cells 1–5 first (setup). HMDB51 is the `easonlll/hmdb51` dataset, which ships
as **extracted frames** (each clip is a folder of JPGs) — the loader handles that
automatically. Note: this mirror has no official split files, so HMDB51 uses a
deterministic random 70/30 split (a known limitation vs UCF101's official splits).

In [ ]:
# ── Cell 8: Get HMDB51 (Drive-cached) ───────────────────────────────────────
import os, glob, shutil, subprocess

H_DIR  = '/content/hmdb51'
H_ZIP  = '/content/drive/MyDrive/hmdb51_dataset.zip'
H_SLUG = 'easonlll/hmdb51'

if not (os.path.isdir(H_DIR) and os.listdir(H_DIR)):
    if not os.path.exists(H_ZIP):
        print('Downloading HMDB51 from Kaggle...')
        subprocess.run(['kaggle', 'datasets', 'download', '-d', H_SLUG, '-p', '/content', '--quiet'], check=True)
        z = glob.glob('/content/hmdb51*.zip')[0]
        print('Caching zip to Drive...')
        shutil.copy(z, H_ZIP)
        zip_to_use = z
    else:
        print('Using cached HMDB51 zip from Drive.')
        zip_to_use = H_ZIP
    os.makedirs(H_DIR, exist_ok=True)
    print('Unzipping HMDB51...')
    subprocess.run(['unzip', '-q', '-o', zip_to_use, '-d', H_DIR], check=True)
print('hmdb51 ready at', H_DIR)
subprocess.run(['ls', H_DIR])

In [ ]:
# ── Cell 9: HMDB51 — train -> benchmark -> Pareto plot ──────────────────────
import subprocess, sys
subprocess.run([sys.executable, 'run_all.py',
    '--data-root', '/content/hmdb51',
    '--dataset', 'hmdb51',
    '--epochs', '10',
    '--batch-size', '8',
], check=True)

In [ ]:
# ── Cell 10: Show the HMDB51 Pareto plot ────────────────────────────────────
from IPython.display import Image, display
display(Image('/content/drive/MyDrive/thesis_results/pareto_hmdb51.png'))